In [2]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
city_urls = {
    "Hyderabad": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Hyderabad",
    "Bangalore": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Bangalore",
    "Mumbai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Mumbai",
    "Pune": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Pune",
    "Chennai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Chennai"
}

# Define headers to prevent a browser from blocking
headers = {'User-Agent': 'Mozilla/5.0'}

max_properties_per_city = 800

# list to store all scraped data
all_data = []

# Keywords to detect actual property types from title text
property_keywords = ['Apartment', 'Flat', 'Studio', 'Villa', 'House', 'Floor', 'Penthouse', 'Row House', 'Farm']

# Loop through each city and its corresponding URL
for city, base_url in city_urls.items():
    print(f"\nScraping {city}...")
    city_data = []   # Store city-specific listings
    page = 1      # Start from page 1

    # Loop through pages until desired property count is reached
    while len(city_data) < max_properties_per_city:
        url = f"{base_url}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Find all property listing cards on the page
        listings = soup.find_all("div", class_="mb-srp__card")
        if not listings:
            print(" No listings found, stopping.")
            break

        # Loop through each property card
        for listing in listings:
            try:
                # Extract bhk from title
                title_tag = listing.find("h2", class_="mb-srp__card--title")
                title = title_tag.get_text(strip=True)

                bhk_match = re.search(r"(\d+)\s*BHK", title.upper())
                bhk = bhk_match.group(1) if bhk_match else None

                # Extract location
                locality = title.split("in", 1)[-1].strip() if "in" in title else None

                # Detect property type from title
                title_clean = title.replace(",", " ").replace("-", " ")
                title_words = title_clean.split()
                prop_type_from_title = next((word for word in title_words if word.capitalize() in property_keywords), None)
            except:
                bhk = locality = prop_type_from_title = None

            # Extract rent price
            try:
                rent = listing.find("div", class_="mb-srp__card__price--amount").get_text(strip=True)
                rent = rent.replace("₹", "").replace(",", "").strip()
            except:
                rent = None

            summary = listing.find("div", class_="mb-srp__card__summary__list")
            summary_items = summary.find_all("div", class_="mb-srp__card__summary__list--item") if summary else []

            # Initializing Variable
            area = furnishing = facing = property_type = overlooking = Bathroom = Balcony=TenantPreferred=Availability= None
            
            # Loop through summary items and extract specific features
            for item in summary_items:
                try:
                    label = item.find("div", class_="mb-srp__card__summary--label").get_text(strip=True).lower()
                    value = item.find("div", class_="mb-srp__card__summary--value").get_text(strip=True)

                    if "area" in label:
                        area = value
                    elif "furnish" in label:
                        furnishing = value
                    elif "facing" in label:
                        facing = value
                    elif "property type" in label or "type" in label:
                        property_type = value
                    elif "overlooking" in label :
                        overlooking = value
                    elif "bathroom" in label:
                        Bathroom = value  
                    elif "balcony" in label:
                        Balcony = value 
                    elif "tenant preferred" in label:
                        TenantPreferred=value
                    elif "availability" in label:
                        Availability=value
                        

                except:
                    continue

            # Final fallback if property_type is missing or invalid
            if not property_type or property_type.lower() in ["for", "in", "rent", "sale"]:
                property_type = prop_type_from_title

            if not bhk or not locality:
                continue

            city_data.append({
                "City": city,
                "BHK": bhk.strip(),
                "Location": locality.strip(),
                "Price (₹)": rent,
                "Area (sqft)": area.strip() if area else None,
                "Property Type": property_type.strip() if property_type else None,
                "Furnishing": furnishing.strip() if furnishing else None,
                "Property Facing": facing.strip() if facing else None,
                "overlooking":overlooking.strip() if overlooking else None,
                "Bathroom":Bathroom.strip() if Bathroom else None,
                "Balcony":Balcony.strip() if Balcony else None,
                "Tenant Preferred":TenantPreferred.strip() if TenantPreferred else None,
                "Availability":Availability.strip() if Availability else None,

                
            })

            if len(city_data) >= max_properties_per_city:
                break

        page += 1    # Move to next Page
        time.sleep(1)  # delay to avoid overloading the server


    # Append city data to the master list
    all_data.extend(city_data)
print("\nScraping completed...")


Scraping Hyderabad...

Scraping Bangalore...

Scraping Mumbai...

Scraping Pune...

Scraping Chennai...

Scraping completed...


In [228]:
df = pd.DataFrame(all_data)
df.to_csv("MagicBricksProject_Scraped_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Scraped_Data.csv")


Data saved to MagicBricksProject_Scraped_Data.csv


In [229]:
df

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,4,"Kokapet, Outer Ring Road Hyderabad",2.3 Lac,5030 sqft,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
1,Hyderabad,3,"Aparna Serene Park, Kondapur, Hyderabad",68000,1500 sqft,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
2,Hyderabad,3,"Prestige Tranquil, Kokapet, Outer Ring Road, H...",75000,1361 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,None,None
3,Hyderabad,1,"Kondapur, Hyderabad",15500,800 sqft,Flat,Semi-Furnished,North,Main Road,1,1,Bachelors,Immediately
4,Hyderabad,3,"Creative RVR Udaya Creative, Kondapur, Hyderabad",55000,2000 sqft,Flat,Semi-Furnished,East,Main Road,3,3,Bachelors/Family,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,Chennai,2,Thazhambur Chennai,20000,810 sqft,House,Semi-Furnished,None,None,2,None,Bachelors/Family,Immediately
3996,Chennai,2,Moolakadai Chennai,23000,1500 sqft,House,Semi-Furnished,None,None,3,1,Bachelors,Immediately
3997,Chennai,2,Mogappair Chennai,15000,1800 sqft,House,Unfurnished,East,None,2,None,Bachelors/Family,Immediately
3998,Chennai,2,Urapakkam Chennai,15000,800 sqft,House,Furnished,None,None,2,None,Bachelors/Family,Immediately


In [274]:
# Save to CSV
df = pd.DataFrame(all_data)
df.to_csv("MagicBricksProject_Scraped_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Scraped_Data.csv")


Data saved to MagicBricksProject_Scraped_Data.csv


In [275]:

# prints number of columns
print("Number of columns:",df.shape[1])

Number of columns: 13


In [276]:

# prints number of rows
print("Number of rows:",df.shape[0])

Number of rows: 4000


In [277]:
# Print the data types of each column
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                object
BHK                 object
Location            object
Price (₹)           object
Area (sqft)         object
Property Type       object
Furnishing          object
Property Facing     object
overlooking         object
Bathroom            object
Balcony             object
Tenant Preferred    object
Availability        object
dtype: object


In [278]:
# Print the number of missing values in each column
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                   0
BHK                    0
Location               0
Price (₹)              0
Area (sqft)            0
Property Type          0
Furnishing            15
Property Facing      919
overlooking         1253
Bathroom              10
Balcony             1076
Tenant Preferred       4
Availability           4
dtype: int64


In [279]:
# It shows the duplicated rows count
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 2


In [280]:
df.drop_duplicates(subset=['City', 'BHK', 'Location', 'Price (₹)', 'Area (sqft)', 'Property Type', 'Furnishing', 'Property Facing'
,'overlooking','Bathroom','Balcony','Tenant Preferred','Availability'], inplace=True)

In [281]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [282]:
print("Number of rows:",df.shape[0])

Number of rows: 3998


In [283]:
# List of locations 
localities = [
    "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
    "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
    "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
    "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
    "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
    "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
    "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
    "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
    "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    # ---------------- PUNE ----------------
    "Shivaji Nagar", "Deccan Gymkhana", "Camp", "Swargate", "Kothrud", "Aundh",
    "Baner", "Bavdhan", "Pashan", "Warje", "Viman Nagar", "Koregaon Park",
    "Kalyani Nagar", "Kharadi", "Hadapsar", "Wagholi", "Mundhwa", "Bibwewadi",
    "Katraj", "Dhayari", "Hinjewadi", "Wakad", "Pimple Saudagar", "Pimpri",

    # ---------------- HYDERABAD ----------------
    "Charminar", "Malakpet", "Nampally", "Asif Nagar", "Gachibowli",
    "HITEC City", "Madhapur", "Kondapur", "Financial District", "Nanakramguda",
    "Kukatpally", "Miyapur", "Chanda Nagar", "Hafeezpet", "Bachupally",
    "Nizampet", "Manikonda", "Narsingi", "Uppal", "Begumpet",
    "Banjara Hills", "Jubilee Hills", "Punjagutta", "Secunderabad",

    # ---------------- MUMBAI ----------------
    "Andheri", "Andheri East", "Andheri West", "Bandra", "Bandra East", "Bandra West",
    "Borivali", "Borivali East", "Borivali West", "Dadar", "Dadar East", "Dadar West",
    "Goregaon", "Goregaon East", "Goregaon West", "Juhu", "Kandivali",
    "Kandivali East", "Kandivali West", "Malad", "Malad East", "Malad West",
    "Lower Parel", "Worli", "Powai", "Chembur", "Ghatkopar", "Kurla",
    "Thane", "Navi Mumbai", "Vashi", "Nerul", "Kharghar",

    # ---------------- CHENNAI ----------------
    "T Nagar", "Adyar", "Velachery", "Anna Nagar", "Tambaram", "Chromepet",
    "Guindy", "Porur", "Vadapalani", "Nungambakkam", "Mylapore", "Kodambakkam",
    "Royapettah", "Egmore", "Triplicane", "Thiruvanmiyur", "Perungudi",
    "Sholinganallur", "OMR", "ECR", "Pallavaram", "Medavakkam",

    # ---------------- BANGALORE ----------------
    "Whitefield", "Marathahalli", "KR Puram", "Indiranagar", "Koramangala",
    "HSR Layout", "BTM Layout", "Electronic City", "Bellandur", "Sarjapur Road",
    "MG Road", "Jayanagar", "JP Nagar", "Banashankari", "Yelahanka",
    "Hebbal", "Malleshwaram", "Rajajinagar", "Basavanagudi", "Kengeri",
    "Nagarbhavi", "Vijayanagar"
]

# Lowercased for comparison
localities_lower = [loc.lower() for loc in localities]

# Function to clean location
def clean_location(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'\s+', ' ', text).lower()
    for loc, loc_lower in zip(localities, localities_lower):
        if loc_lower in text:
            return loc
    return np.nan

# Update Location column in-place
df['Location'] = df['Location'].apply(clean_location)  

In [284]:
# renaming the column
df.rename(columns={"Price (₹)": "Price"}, inplace=True)

In [285]:
# Function to clean and convert price to integer
def clean_price(price):
    price = str(price).replace(',', '').strip().lower()

    if 'lac' in price:
        num = float(price.split()[0])
        return int(round(num * 100000))
    elif 'cr' in price:
        num = float(price.split()[0])
        return int(round(num * 10000000))
    else:
        return int(''.join(filter(str.isdigit, price)))
    
# Apply cleaning function to 'Price' column
df["Price"] = df["Price"].apply(clean_price)

In [286]:

# Extract numeric part from 'Area (sqft)' and convert to integer
df["Area (sqft)"] = df["Area (sqft)"].str.extract(r'(\d+)', expand=False).astype(float).astype(int)

In [287]:
df.head(10)

,City,BHK,Location,Price,Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,4,Kokapet,230000,5030,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
1,Hyderabad,3,Kondapur,68000,1500,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
2,Hyderabad,3,Kokapet,75000,1361,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,None,None
3,Hyderabad,1,Kondapur,15500,800,Flat,Semi-Furnished,North,Main Road,1,1,Bachelors,Immediately
4,Hyderabad,3,Kondapur,55000,2000,Flat,Semi-Furnished,East,Main Road,3,3,Bachelors/Family,Immediately
5,Hyderabad,3,Kokapet,85000,2124,Flat,Semi-Furnished,East,Main Road,3,1,Bachelors,Immediately
6,Hyderabad,3,Kondapur,58000,1750,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,1,Bachelors,Immediately
7,Hyderabad,3,Kokapet,70000,2020,Flat,Semi-Furnished,West,"Garden/Park, Pool",3,1,Bachelors/Family,Immediately
8,Hyderabad,3,Gachibowli,85000,1796,Flat,Semi-Furnished,East,Garden/Park,3,1,Bachelors,Immediately
9,Hyderabad,3,NaN,75000,2100,Flat,Semi-Furnished,North - East,Garden/Park,3,3,Bachelors/Family,Immediately


In [288]:
# Replace empty strings in 'Location' with NaN
df['Location'] = df['Location'].replace('', np.nan)

In [289]:
# Fill missing 'Location' values with the most frequent location (mode) within each city
df['Location'] = df.groupby('City')['Location'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)

In [290]:
# Count remaining missing values in 'Location' column
df['Location'].isna().sum()

np.int64(0)

In [291]:
# Fill missing 'Property Facing' values with the most frequent location (mode) within each city
df['Property Facing'] = df.groupby('Location')['Property Facing'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'East')
)

In [292]:

# Count remaining missing values in 'Location' column
df['Property Facing'].isna().sum()

np.int64(0)

In [293]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 2


In [294]:
df.drop_duplicates(subset=['City', 'BHK', 'Location', 'Price', 'Area (sqft)', 'Property Type', 'Furnishing', 'Property Facing'
,'overlooking','Bathroom','Balcony','Tenant Preferred','Availability'], inplace=True)

In [295]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [296]:
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                   0
BHK                    0
Location               0
Price                  0
Area (sqft)            0
Property Type          0
Furnishing            15
Property Facing        0
overlooking         1250
Bathroom              10
Balcony             1073
Tenant Preferred       4
Availability           4
dtype: int64


In [297]:
# Fill missing 'Furnishing' values with the most frequent location (mode) within each city
df['Furnishing'] = df['Furnishing'].fillna(df['Furnishing'].mode()[0])

In [298]:
df["overlooking"]=df["overlooking"].fillna(df["overlooking"].mode()[0])

In [299]:
df["Bathroom"]=df["Bathroom"].fillna(df["Bathroom"].mode()[0])

In [300]:
df["Tenant Preferred"]=df["Tenant Preferred"].fillna(df["Tenant Preferred"].mode()[0])

In [301]:
df['Availability']=df['Availability'].fillna(df['Availability'].mode()[0])

In [302]:
df['Balcony']=df['Balcony'].fillna(df['Balcony'].mode()[0])

In [303]:

print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                0
BHK                 0
Location            0
Price               0
Area (sqft)         0
Property Type       0
Furnishing          0
Property Facing     0
overlooking         0
Bathroom            0
Balcony             0
Tenant Preferred    0
Availability        0
dtype: int64


In [304]:
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                object
BHK                 object
Location            object
Price                int64
Area (sqft)          int64
Property Type       object
Furnishing          object
Property Facing     object
overlooking         object
Bathroom            object
Balcony             object
Tenant Preferred    object
Availability        object
dtype: object


In [305]:

df['BHK'] = df['BHK'].astype(int)


In [306]:
df

,City,BHK,Location,Price,Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,4,Kokapet,230000,5030,Villa,Semi-Furnished,East,Garden/Park,5,1,Bachelors/Family,Immediately
1,Hyderabad,3,Kondapur,68000,1500,Flat,Semi-Furnished,West,Main Road,3,1,Bachelors/Family,Immediately
2,Hyderabad,3,Kokapet,75000,1361,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,2,Bachelors/Family,Immediately
3,Hyderabad,1,Kondapur,15500,800,Flat,Semi-Furnished,North,Main Road,1,1,Bachelors,Immediately
4,Hyderabad,3,Kondapur,55000,2000,Flat,Semi-Furnished,East,Main Road,3,3,Bachelors/Family,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,Chennai,2,Anna Nagar,20000,810,House,Semi-Furnished,East,Main Road,2,1,Bachelors/Family,Immediately
3996,Chennai,2,Anna Nagar,23000,1500,House,Semi-Furnished,East,Main Road,3,1,Bachelors,Immediately
3997,Chennai,2,Anna Nagar,15000,1800,House,Unfurnished,East,Main Road,2,1,Bachelors/Family,Immediately
3998,Chennai,2,Anna Nagar,15000,800,House,Furnished,East,Main Road,2,1,Bachelors/Family,Immediately


In [321]:
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                object
BHK                  int64
Location            object
Price                int64
Area (sqft)          int64
Property Type       object
Furnishing          object
Property Facing     object
overlooking         object
Bathroom             int64
Balcony              int64
Tenant Preferred    object
Availability        object
dtype: object


In [308]:
print(df['Price'].describe())
print(df['Area (sqft)'].describe())
print(df['BHK'].describe())
print(df['Bathroom'].describe())
print(df['Balcony'].describe())

count    3.996000e+03
mean     1.053894e+05
std      1.052398e+06
min      1.000000e+03
25%      2.500000e+04
50%      4.000000e+04
75%      7.500000e+04
max      5.500000e+07
Name: Price, dtype: float64
count     3996.000000
mean      1350.199700
std       1049.622489
min          3.000000
25%        800.000000
50%       1100.000000
75%       1600.000000
max      28000.000000
Name: Area (sqft), dtype: float64
count    3996.000000
mean        2.508258
std         0.901220
min         1.000000
25%         2.000000
50%         2.000000
75%         3.000000
max        10.000000
Name: BHK, dtype: float64
count     3996
unique       9
top          2
freq      1870
Name: Bathroom, dtype: object
count     3996
unique       8
top          1
freq      2345
Name: Balcony, dtype: object


In [313]:
print(df['Bathroom'].unique())
print(df['Balcony'].unique())

# Clean 'Bathroom' and 'Balcony' by removing non-numeric characters and converting to numeric
df['Bathroom'] = pd.to_numeric(df['Bathroom'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')
df['Balcony'] = pd.to_numeric(df['Balcony'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')

# Fill missing values with the median
df['Bathroom'] = df['Bathroom'].fillna(df['Bathroom'].median())
df['Balcony'] = df['Balcony'].fillna(df['Balcony'].median())

# Handle outliers by capping at the 99th percentile
df['Bathroom'] = df['Bathroom'].clip(upper=df['Bathroom'].quantile(0.99))
df['Balcony'] = df['Balcony'].clip(upper=df['Balcony'].quantile(0.99))



[5 3 1 2 4 6]
[1 2 3 4]


In [323]:
df.to_csv("MagicBricksProject_Cleaned_Data.csv", index=False)